[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/duckdb-certified/notebooks/day-02-reading-files.ipynb#scrollTo=1a2b3c4d)

---
# Day 2 · Reading Files Directly — Parquet, CSV, and JSON
**certified-journeys / duckdb-certified** · Day 2 · File Formats

> **Goal for today:** Query CSV, Parquet, and JSON files directly without loading them into tables, combine multi-file datasets with glob patterns, and materialise filtered subsets efficiently.


In [ ]:
%pip install -q duckdb pandas pyarrow


## Setup · Create Sample Files

This notebook generates all sample files locally so every cell runs in a fresh Colab environment with no external dependencies.

We'll create:
- `data/orders.csv` — 500,000 rows of order data
- `data/products.json` — product catalogue with nested fields
- `data/sales_2022.parquet`, `data/sales_2023.parquet` — annual sales Parquet shards

> **Reading:** [DuckDB Data Import Overview](https://duckdb.org/docs/data/overview)


In [ ]:
import os
import json
import random
import datetime
import duckdb
import pandas as pd

os.makedirs("data", exist_ok=True)

# ── orders.csv ──────────────────────────────────────────────────────────────
random.seed(42)
regions   = ["North", "South", "East", "West"]
statuses  = ["completed", "pending", "cancelled"]
base_date = datetime.date(2023, 1, 1)

with open("data/orders.csv", "w") as f:
    f.write("order_id,customer_id,order_date,region,status,amount\n")
    for i in range(1, 500_001):
        d = base_date + datetime.timedelta(days=random.randint(0, 364))
        amt = round(random.uniform(10, 1000), 2)
        f.write(f"{i},{random.randint(1,10000)},{d},{random.choice(regions)},{random.choice(statuses)},{amt}\n")

print("✓ orders.csv written (500 000 rows)")

# ── products.json (NDJSON — one JSON object per line) ───────────────────────
categories = ["Electronics", "Clothing", "Home", "Sports"]
with open("data/products.json", "w") as f:
    for pid in range(1, 201):
        rec = {
            "product_id": pid,
            "name": f"Product {pid}",
            "category": random.choice(categories),
            "price": round(random.uniform(5, 500), 2),
            "tags": random.sample(["sale", "new", "bestseller", "clearance"], k=random.randint(0, 2)),
            "meta": {"weight_kg": round(random.uniform(0.1, 10), 2), "in_stock": random.choice([True, False])}
        }
        f.write(json.dumps(rec) + "\n")

print("✓ products.json written (200 products, NDJSON)")

# ── Parquet shards using DuckDB itself ──────────────────────────────────────
con = duckdb.connect()
for year in [2022, 2023]:
    con.execute(f"""
        COPY (
            SELECT
                i                                            AS sale_id,
                date '{year}-01-01' + INTERVAL (i % 365) DAY AS sale_date,
                ['North','South','East','West'][1 + (i % 4)] AS region,
                round(random() * 990 + 10, 2)                AS amount
            FROM range(1, 300_001) t(i)
        ) TO 'data/sales_{year}.parquet' (FORMAT PARQUET)
    """)

print("✓ sales_2022.parquet and sales_2023.parquet written (300 000 rows each)")
print("\nAll sample files ready.")


## Step 1 · Querying CSV Directly with `read_csv_auto()`

DuckDB can query a CSV file as if it were a table — **no `LOAD` step, no staging table, no schema definition needed**.

`read_csv_auto()` (alias: `read_csv()`) inspects a sample of the file to infer:
- Column names (from header row)
- Data types (`VARCHAR`, `BIGINT`, `DATE`, `DOUBLE`, …)
- Delimiter, quoting, newline style

You can override any inferred setting:
```sql
read_csv('file.csv', sep=';', header=false, columns={'col1': 'INT', 'col2': 'VARCHAR'})
```

> **Reading:** [DuckDB CSV Import](https://duckdb.org/docs/data/csv/overview)


In [ ]:
# Query CSV directly — DuckDB treats the file path as a table expression
result = con.execute("""
    SELECT
        region,
        status,
        COUNT(*)                  AS order_count,
        round(SUM(amount), 2)     AS total_revenue,
        round(AVG(amount), 2)     AS avg_order_value
    FROM read_csv_auto('data/orders.csv')
    WHERE status = 'completed'
    GROUP BY region, status
    ORDER BY total_revenue DESC
""").df()

print(result.to_string(index=False))

# Inspect the inferred schema — useful for verifying type inference
print("\nInferred schema:")
schema = con.execute("DESCRIBE SELECT * FROM read_csv_auto('data/orders.csv') LIMIT 0").fetchall()
for col in schema:
    print(f"  {col[0]:15s} → {col[1]}")


### What just happened?
- **`read_csv_auto('path')`** is a table-valued function — it appears wherever a table name can appear: `FROM`, `JOIN`, subqueries.
- DuckDB streamed the 500 000-row file without ever loading it entirely into memory; predicate pushdown (`WHERE status = 'completed'`) filtered rows as they were read.
- **`DESCRIBE SELECT * FROM read_csv_auto(...) LIMIT 0`** is a zero-cost schema inspection trick — the `LIMIT 0` prevents any data from being read.
- Type inference correctly identified `order_date` as `DATE` and `amount` as `DOUBLE` from the header and first few rows.


## Step 2 · Reading Parquet Files with `read_parquet()`

Parquet is DuckDB's **native format** — it stores data in the same columnar layout DuckDB uses internally. This means:
- Only the columns referenced in your query are read from disk
- Row group statistics (min/max) enable DuckDB to skip entire row groups
- No deserialization overhead — Parquet pages map directly to DuckDB vectors

**Parquet file layout:**
```
File
 └─ Row groups (default: ~128 MB each)
     └─ Column chunks (one per column per row group)
         └─ Pages (compressed data, ~8 KB each)
```

> **Reading:** [DuckDB Parquet Support](https://duckdb.org/docs/data/parquet/overview)


In [ ]:
# Read a single Parquet file — syntax identical to read_csv_auto
parquet_info = con.execute("""
    SELECT
        year(sale_date)          AS yr,
        region,
        COUNT(*)                 AS sales_count,
        round(SUM(amount), 2)    AS total
    FROM read_parquet('data/sales_2023.parquet')
    GROUP BY yr, region
    ORDER BY yr, region
""").df()

print(parquet_info.to_string(index=False))

# Inspect Parquet file metadata — row groups, column statistics
meta = con.execute("SELECT * FROM parquet_metadata('data/sales_2023.parquet')").df()
print(f"\nRow groups: {len(meta)}")
print(f"Total rows: {meta['total_uncompressed_size'].sum():,} bytes (uncompressed)")

# Column statistics per row group — used for predicate pushdown
col_meta = con.execute("""
    SELECT column_name, stats_min, stats_max, num_values
    FROM parquet_column_metadata('data/sales_2023.parquet')
    WHERE column_name = 'amount'
""").df()
print("\nColumn stats for 'amount':")
print(col_meta.head(5).to_string(index=False))


### What just happened?
- **`read_parquet('path')`** reads a Parquet file; DuckDB reads only the `sale_date`, `region`, and `amount` columns — it skipped `sale_id` entirely.
- **`parquet_metadata()`** exposes row group statistics: DuckDB uses `stats_min`/`stats_max` to skip row groups that can't satisfy a `WHERE` clause.
- **`parquet_column_metadata()`** shows per-column compression and encoding choices — useful when optimising write-time settings.
- The `year()` function is applied in-engine — no Python-side date parsing needed.


## Step 3 · Querying JSON with `read_json_auto()` and Dot-Notation

DuckDB supports **NDJSON** (newline-delimited JSON) natively. It auto-infers schema from the first sample of lines and exposes nested fields via dot-notation and bracket syntax:

| Access pattern | Example |
|---|---|
| Top-level field | `category` |
| Nested struct | `meta.weight_kg` |
| Nested struct (bracket) | `meta['in_stock']` |
| Array element | `tags[1]` (1-indexed) |
| Array length | `array_length(tags)` |
| Unnest array | `UNNEST(tags)` |

> **Reading:** [DuckDB JSON Support](https://duckdb.org/docs/data/json/overview)


In [ ]:
# Query JSON file and access nested fields via dot-notation
df_json = con.execute("""
    SELECT
        product_id,
        name,
        category,
        price,
        meta.weight_kg           AS weight_kg,      -- nested struct access
        meta.in_stock            AS in_stock,
        array_length(tags)       AS tag_count        -- array function
    FROM read_json_auto('data/products.json')
    WHERE meta.in_stock = true
      AND price > 200
    ORDER BY price DESC
    LIMIT 8
""").df()

print(df_json.to_string(index=False))

# UNNEST to explode the tags array into individual rows
print("\nTop 5 most common tags:")
tag_counts = con.execute("""
    SELECT tag, COUNT(*) AS freq
    FROM (
        SELECT UNNEST(tags) AS tag
        FROM read_json_auto('data/products.json')
        WHERE array_length(tags) > 0
    )
    GROUP BY tag
    ORDER BY freq DESC
""").fetchall()
for tag, freq in tag_counts:
    print(f"  {tag:15s}  {freq}")


### What just happened?
- **`meta.weight_kg`** — DuckDB promotes JSON nested objects to `STRUCT` types; dot-notation traverses them without any `json_extract()` call.
- **`UNNEST(tags)`** explodes the `tags` array into one row per element — equivalent to `explode()` in PySpark or Polars.
- **`array_length(tags)`** works on the `LIST` type that DuckDB infers for JSON arrays.
- DuckDB read only the fields referenced in the query — if the JSON has 50 fields but you SELECT 4, only those 4 are materialised.


## Step 4 · Multi-File Queries with Glob Patterns

Real datasets are almost never a single file. DuckDB accepts **glob patterns** in `read_parquet()` and `read_csv_auto()` to union multiple files in one query:

| Pattern | Matches |
|---|---|
| `data/*.parquet` | All Parquet files in `data/` |
| `data/sales_202*.parquet` | Files starting with `sales_202` |
| `data/**/*.parquet` | All Parquet files in any subdirectory |
| `['f1.parquet', 'f2.parquet']` | Explicit list of files |

**`filename=true`** adds a `filename` column so you can tell which file each row came from — essential for debugging multi-file ingests.


In [ ]:
# Combine both annual Parquet shards with a single glob pattern
print("=== Combined 2022 + 2023 sales via glob ===")
combined = con.execute("""
    SELECT
        year(sale_date)       AS yr,
        COUNT(*)              AS rows,
        round(SUM(amount), 2) AS total_revenue
    FROM read_parquet('data/sales_*.parquet')
    GROUP BY yr
    ORDER BY yr
""").df()
print(combined.to_string(index=False))

# filename=true — adds the source filename as a column
print("\n=== First 3 rows from each file (filename=true) ===")
with_source = con.execute("""
    SELECT filename, sale_id, sale_date, amount
    FROM read_parquet('data/sales_*.parquet', filename=true)
    LIMIT 6
""").df()
print(with_source.to_string(index=False))

# Explicit list syntax — useful when you need non-glob selection
print("\n=== Explicit list ===")
explicit = con.execute("""
    SELECT COUNT(*) AS total_rows
    FROM read_parquet(['data/sales_2022.parquet', 'data/sales_2023.parquet'])
""").fetchone()[0]
print(f"Total rows across both files: {explicit:,}")


### What just happened?
- **`read_parquet('data/sales_*.parquet')`** unions all matching files; DuckDB reads them in parallel using its multi-threaded executor.
- **`filename=true`** injects the source path as an extra column — critical for debugging which shard a bad row came from.
- The **explicit list** form `read_parquet(['f1.parquet', 'f2.parquet'])` is useful when the files don't share a simple glob pattern.
- DuckDB uses **file-level metadata** to prune files before reading — if a Parquet file's row-group stats show no rows matching a `WHERE year = 2022` predicate, that file is skipped entirely.


## Step 5 · JOINing Files Without Materialising Either

**The key DuckDB productivity unlock:** you can JOIN a CSV against a Parquet file (or JSON against CSV) in a single query — neither file needs to be loaded into a table first.

This is analogous to `dbt` model composition, except the query runs entirely in-process with zero staging overhead.

```sql
-- Production pattern: join a CSV and a Parquet file
SELECT o.order_id, p.category, o.amount
FROM read_csv_auto('orders.csv')    AS o
JOIN read_parquet('products.parquet') AS p ON o.product_id = p.product_id
```


In [ ]:
# Add a product_id column to the orders CSV so we can JOIN
# (In practice, your files already share a key — we simulate it here)
random.seed(0)
with open("data/orders_with_product.csv", "w") as f:
    f.write("order_id,customer_id,order_date,region,status,product_id,amount\n")
    for i in range(1, 50_001):
        d = base_date + datetime.timedelta(days=random.randint(0, 364))
        amt = round(random.uniform(10, 1000), 2)
        f.write(f"{i},{random.randint(1,10000)},{d},{random.choice(regions)},"
                f"{random.choice(statuses)},{random.randint(1,200)},{amt}\n")

# JOIN CSV against JSON — no tables involved
joined = con.execute("""
    SELECT
        p.category,
        o.region,
        COUNT(*)              AS order_count,
        round(SUM(o.amount), 2) AS revenue
    FROM read_csv_auto('data/orders_with_product.csv') AS o
    JOIN read_json_auto('data/products.json')          AS p
      ON o.product_id = p.product_id
    GROUP BY p.category, o.region
    ORDER BY revenue DESC
    LIMIT 10
""").df()

print("Revenue by category and region (CSV × JSON join):")
print(joined.to_string(index=False))


### What just happened?
- DuckDB used **hash-join** on `product_id` — it built a hash table from the smaller JSON file (200 rows) and probed it with each CSV row.
- Neither file was loaded into a named table — the entire pipeline ran as a single query plan.
- **In production**, you'd replace file paths with table names after `ATTACH`-ing an external data source, but the query structure is identical.
- DuckDB's query planner automatically chose the correct join build/probe sides based on estimated cardinality.


## Step 6 · Materialising Filtered Subsets with `CREATE TABLE AS SELECT`

`CREATE TABLE AS SELECT` (CTAS) materialises a query result into a DuckDB table. Use it when:
- A subset of a large file is queried repeatedly (avoid re-reading the file on every query)
- You want columnar compression benefits on a filtered slice
- You need to JOIN the result against other tables multiple times

**`COPY ... TO`** writes the result back to a file (CSV or Parquet) — useful for publishing derived datasets.

```sql
-- Materialise to table
CREATE TABLE my_table AS SELECT ... FROM read_csv_auto('big_file.csv') WHERE ...

-- Write directly to file
COPY (SELECT ... FROM read_csv_auto('input.csv') WHERE ...) TO 'output.parquet' (FORMAT PARQUET)
```


In [ ]:
import time

# ── CTAS: materialise completed orders from 2023 into a DuckDB table ─────────
con.execute("DROP TABLE IF EXISTS completed_orders")
con.execute("""
    CREATE TABLE completed_orders AS
    SELECT *
    FROM read_csv_auto('data/orders.csv')
    WHERE status = 'completed'
""")

n_materialised = con.execute("SELECT COUNT(*) FROM completed_orders").fetchone()[0]
print(f"Materialised {n_materialised:,} completed orders into DuckDB table")

# Compare query time: file scan vs table scan
query = "SELECT region, round(AVG(amount), 2) AS avg_amount FROM {} GROUP BY region"

start = time.perf_counter()
con.execute(query.format("read_csv_auto('data/orders.csv') WHERE status = 'completed'")).fetchall()
csv_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
con.execute(query.format("completed_orders")).fetchall()
table_ms = (time.perf_counter() - start) * 1000

print(f"\nCSV file scan : {csv_ms:.1f} ms")
print(f"DuckDB table  : {table_ms:.1f} ms")

# ── COPY TO: write a filtered Parquet shard ──────────────────────────────────
con.execute("""
    COPY (
        SELECT order_id, order_date, region, amount
        FROM completed_orders
        WHERE amount > 500
    ) TO 'data/high_value_orders.parquet' (FORMAT PARQUET)
""")

hv_count = con.execute("SELECT COUNT(*) FROM read_parquet('data/high_value_orders.parquet')").fetchone()[0]
print(f"\nHigh-value orders exported to Parquet: {hv_count:,} rows")


### What just happened?
- **CTAS** filtered 500 000 CSV rows down to `~166 000` completed orders and stored them in DuckDB's columnar format.
- Subsequent queries on the materialised table are faster because DuckDB doesn't need to parse CSV, re-infer types, or filter again.
- **`COPY ... TO 'file.parquet'`** writes a compressed Parquet file — the production pattern for publishing derived datasets to object storage (S3, GCS) or sharing with downstream consumers.
- The speed difference grows with query complexity — the more filtering and aggregation, the bigger the win for materialisation.


In [ ]:
# Challenge: Multi-format query pipeline
#
# Using the files created in this notebook:
#   - data/orders_with_product.csv  (orders with product_id)
#   - data/products.json            (product catalogue with category + price)
#
# Write a query that:
#   1. JOINs orders (CSV) against products (JSON) on product_id
#   2. Filters to category = 'Electronics' and status = 'completed'
#   3. Computes per-region: total_orders, total_revenue, avg_product_price
#   4. Uses CREATE TABLE AS SELECT to materialise the result as 'electronics_summary'
#   5. Then exports electronics_summary to 'data/electronics_summary.parquet'
#
# Bonus: use the filename=true trick to verify which source files contributed

# Your solution here
# con.execute("""
#     CREATE TABLE electronics_summary AS
#     SELECT ...
#     FROM read_csv_auto('data/orders_with_product.csv') AS o
#     JOIN ...
# """)


---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| `read_csv_auto()` | Queries CSV in-place; auto-infers schema; accepts override options |
| `read_parquet()` | Native columnar format; column + row-group pruning; fastest for analytics |
| `read_json_auto()` | Infers struct types; dot-notation for nested fields; `UNNEST` for arrays |
| Glob patterns | `'data/*.parquet'` unions all matching files; `filename=true` adds source column |
| Cross-format JOIN | JOIN CSV against Parquet/JSON without materialising either |
| CTAS | Materialise filtered subsets for repeated queries; `COPY TO` writes back to files |

> **Tip:** DuckDB treats file paths as table names — you can JOIN a CSV against a Parquet file in a single query without materialising either. This is the key productivity unlock.

---
## What's next
**Day 3** → Advanced SQL in DuckDB — window functions, CTEs, PIVOT, and reusable macros.

Mark Day 2 complete in your [tracker](../index.html).
